In [2]:
! pip install llama-index llama-index-llms-mistralai llama-index-embeddings-mistralai mistralai

Defaulting to user installation because normal site-packages is not writeable


In [7]:
import nltk
print(nltk.__version__)

3.9.1


In [8]:
! pip show nltk

Name: nltk
Version: 3.9.1
Summary: Natural Language Toolkit
Home-page: https://www.nltk.org/
Author: NLTK Team
Author-email: nltk.team@gmail.com
License: Apache License, Version 2.0
Location: C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages
Requires: click, joblib, regex, tqdm
Required-by: llama-index, llama-index-core


In [23]:
from llama_index.core import Settings, VectorStoreIndex, Document
from llama_index.readers.file import PDFReader
from llama_index.llms.mistralai import MistralAI
from llama_index.embeddings.mistralai import MistralAIEmbedding
from getpass import getpass

# --- STEP 1: API Key ---
api_key = getpass("Enter your Mistral API key: ")

# --- STEP 2: Load PDF ---
pdf_path = "example_1_compressed.pdf"
reader = PDFReader()
all_docs = reader.load_data(pdf_path)

# --- STEP 3: Add line numbers manually ---
docs = []
for doc in all_docs:
    page_label = (
        doc.metadata.get("page_label")
        or doc.metadata.get("page")
        or doc.metadata.get("page_number")
        or 0
    )
    try:
        page_num = int(page_label)
    except ValueError:
        page_num = 0

    if 5 <= page_num <= 15:  # ✅ only pages 5–15
        lines = doc.text.split("\n")
        for i, line in enumerate(lines, start=1):
            if line.strip():
                docs.append(
                    Document(
                        text=line,
                        metadata={"page": page_num, "line": i}
                    )
                )

print(f"Loaded {len(docs)} lines from pages 5–15.")

# --- STEP 4: Setup Models ---
Settings.llm = MistralAI(model="mistral-large-latest", api_key=api_key, max_retries=5)
Settings.embed_model = MistralAIEmbedding(model_name="mistral-embed", api_key=api_key)

# --- STEP 5: Build Index ---
index = VectorStoreIndex.from_documents(docs)

# --- STEP 6: Query Engine ---
query_engine = index.as_query_engine(similarity_top_k=5)

# --- STEP 7: Prompt for TOC with page + line ---
prompt = """
Read the PDF transcript (pages 5–15).
Generate a structured Table of Contents (TOC).
For each major section (e.g., OPENING STATEMENT, STATEMENT OF, PREPARED STATEMENT):
- Give the section title
- Include the page number
- Include the starting line number where that section begins
- Keep chronological order

Format:
1. [Section Title] — Page X, Line Y
2. [Section Title] — Page Z, Line A
"""

response = query_engine.query(prompt)

print("\n--- Generated TOC (Pages 5–15, with line numbers) ---\n")
print(str(response))


Loaded 587 lines from pages 5–15.


2025-08-25 23:10:43,894 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2025-08-25 23:10:44,728 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2025-08-25 23:10:45,431 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2025-08-25 23:10:45,853 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2025-08-25 23:10:46,571 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2025-08-25 23:10:47,082 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2025-08-25 23:10:47,479 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2025-08-25 23:10:48,108 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2025-08-25 23:10:48,515 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2025-08-25 23:10:49,525 - INFO - HTTP

SDKError: API error occurred: Status 429
{"object":"error","message":"Service tier capacity exceeded for this model.","type":"service_tier_capacity_exceeded","param":null,"code":"3505"}